# Ejercicio 3 — Consultas directas sobre archivos Parquet

Exploración inicial de los datos de 2026 **sin importarlos a una tabla**: todas las consultas leen los
archivos de `data/raw/` en el momento de ejecutarse, con `read_parquet`, `glob`, `parquet_metadata` y
`parquet_schema`, o con las vistas `trips` y `trips_clean`, que son consultas guardadas sobre esos mismos
archivos.

Cada consulta está en `sql/03_exploration/` y su documentación (objetivo, fuentes, resultado y decisión)
en `docs/03_exploration.md`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd

import taxi_db

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
con = taxi_db.connect()

## 3.1 Cantidad de archivos

### Inventario de archivos por tipo y año

Cuenta los archivos que coinciden con el patrón `data/raw/*/*/*.parquet` sin abrirlos.

In [2]:
taxi_db.show(con, "03_exploration/01_file_inventory.sql")

SELECT
    split_part(file, '/', 3) AS taxi_type,
    split_part(file, '/', 4) AS year,
    count(*) AS files,
    min(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1)) AS first_month,
    max(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1)) AS last_month
FROM glob('data/raw/*/*/*.parquet')
GROUP BY ALL
ORDER BY ALL;


,taxi_type,year,files,first_month,last_month
0,green,2026,8,2026-01,2026-08
1,yellow,2026,8,2026-01,2026-08


## 3.2 Cantidad de registros

### Registros por archivo según los metadatos

Lee solo el pie (footer) de cada Parquet: número de filas, grupos de filas y tamaños comprimido y sin comprimir.

In [3]:
taxi_db.show(con, "03_exploration/02_rows_per_file.sql")

SELECT
    regexp_extract(file_name, '([a-z]+_tripdata_\d{4}-\d{2})\.parquet$', 1) AS file,
    num_rows,
    num_row_groups,
    round(sum(total_compressed_size) / 1024 / 1024, 1) AS compressed_mib,
    round(sum(total_uncompressed_size) / 1024 / 1024, 1) AS uncompressed_mib
FROM parquet_metadata('data/raw/*/*/*.parquet')
JOIN parquet_file_metadata('data/raw/*/*/*.parquet') USING (file_name)
GROUP BY ALL
ORDER BY file;


,file,num_rows,num_row_groups,compressed_mib,uncompressed_mib
0,green_tripdata_2026-01,40272,1,0.9,1.2
1,green_tripdata_2026-02,37373,1,0.9,1.1
2,green_tripdata_2026-03,44208,1,1.0,1.3
3,green_tripdata_2026-04,44238,1,1.0,1.3
4,green_tripdata_2026-05,44921,1,1.0,1.4
5,green_tripdata_2026-06,44163,1,1.0,1.3
6,green_tripdata_2026-07,41252,1,1.0,1.3
7,green_tripdata_2026-08,40687,1,1.0,1.3
8,yellow_tripdata_2026-01,3724889,4,61.2,94.8
9,yellow_tripdata_2026-02,3399866,4,55.9,85.9


### Conteo de registros leyendo los datos

Cuenta las filas leyendo los archivos, por tipo de taxi y en total, para contrastar con los metadatos.

In [4]:
taxi_db.show(con, "03_exploration/03_total_rows.sql")

SELECT
    coalesce(split_part(filename, '/', 3), 'total') AS taxi_type,
    count(*) AS trips
FROM read_parquet('data/raw/*/*/*.parquet', union_by_name = true, filename = true)
GROUP BY ROLLUP (split_part(filename, '/', 3))
ORDER BY trips;


,taxi_type,trips
0,green,337114
1,yellow,29703355
2,total,30040469


## 3.3 y 3.4 Columnas y tipos de datos

### Columnas presentes en cada tipo de archivo

Muestra en cuántos archivos amarillos y verdes aparece cada columna, desde qué mes y con qué tipo físico Parquet.

In [5]:
taxi_db.show(con, "03_exploration/04_columns_by_file.sql")

SELECT
    name AS column_name,
    count(*) FILTER (WHERE file_name LIKE '%yellow_tripdata%') AS yellow_files,
    count(*) FILTER (WHERE file_name LIKE '%green_tripdata%') AS green_files,
    min(regexp_extract(file_name, '(\d{4}-\d{2})\.parquet$', 1)) AS first_month,
    string_agg(DISTINCT type, ', ') AS physical_types
FROM parquet_schema('data/raw/*/*/*.parquet')
WHERE type IS NOT NULL
GROUP BY name
ORDER BY yellow_files + green_files DESC, column_name;


,column_name,yellow_files,green_files,first_month,physical_types
0,DOLocationID,8,8,2026-01,INT32
1,PULocationID,8,8,2026-01,INT32
2,RatecodeID,8,8,2026-01,INT64
3,VendorID,8,8,2026-01,INT32
4,cbd_congestion_fee,8,8,2026-01,DOUBLE
5,congestion_surcharge,8,8,2026-01,DOUBLE
6,extra,8,8,2026-01,DOUBLE
7,fare_amount,8,8,2026-01,DOUBLE
8,improvement_surcharge,8,8,2026-01,DOUBLE
9,mta_tax,8,8,2026-01,DOUBLE


### Tipos lógicos de los taxis amarillos

Esquema unificado (`union_by_name`) que DuckDB infiere para todos los archivos amarillos.

In [6]:
taxi_db.show(con, "03_exploration/05_column_types_yellow.sql")

DESCRIBE SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true);


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


### Tipos lógicos de los taxis verdes

Esquema unificado de los archivos verdes.

In [7]:
taxi_db.show(con, "03_exploration/06_column_types_green.sql")

DESCRIBE SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


### Columnas con tipos distintos entre archivos

Detecta columnas cuyo tipo físico cambia de un archivo a otro.

In [8]:
taxi_db.show(con, "03_exploration/07_type_conflicts.sql")

SELECT
    name AS column_name,
    string_agg(DISTINCT type, ', ') AS physical_types,
    count(DISTINCT type) AS type_count
FROM parquet_schema('data/raw/*/*/*.parquet')
WHERE type IS NOT NULL
GROUP BY name
HAVING count(DISTINCT type) > 1
ORDER BY column_name;


,column_name,physical_types,type_count


## 3.5 Muestra de registros

### Muestra de taxis amarillos

Muestra reproducible de 5 filas (semilla 42).

In [9]:
taxi_db.show(con, "03_exploration/08_sample_yellow.sql")

SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.9,3.50,0.5,3.20,0.00,1.0,16.10,2.5,0.0,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.2,4.25,0.5,3.95,0.00,1.0,23.90,2.5,0.0,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.2,1.00,0.5,2.44,0.00,1.0,14.64,2.5,0.0,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.7,1.00,0.5,4.69,0.00,1.0,28.14,2.5,0.0,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.0,0.00,0.5,0.00,6.94,1.0,81.69,2.5,0.0,0.75,None


### Muestra de taxis verdes

Muestra reproducible de 5 filas (semilla 42).

In [10]:
taxi_db.show(con, "03_exploration/09_sample_green.sql")

SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.0,0.0,0.0,14.20,0.0,NaN,1.0,85.20,1,2,0.0,0.0,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.9,1.0,0.5,2.08,0.0,NaN,1.0,12.48,1,1,0.0,0.0,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.8,1.0,1.5,0.00,0.0,NaN,1.0,8.30,2,1,0.0,0.0,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.7,2.5,0.5,4.34,0.0,NaN,1.0,26.04,1,1,0.0,0.0,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.6,1.0,0.5,1.00,0.0,NaN,1.0,12.10,1,1,0.0,0.0,None


## 3.6 Problemas de calidad de datos

### Perfil estadístico de los taxis amarillos

`SUMMARIZE` calcula mínimo, máximo, valores únicos aproximados, media, cuartiles y porcentaje de nulos de cada columna.

In [11]:
taxi_db.show(con, "03_exploration/10_summary_yellow.sql")

SUMMARIZE SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true);


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.885739809526567,0.7155277514022341,2,2,2,29703355,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-30 15:29:29.519432,NaN,2026-03-03 16:02:46.765604,2026-04-30 21:25:12.847281,2026-06-26 08:00:11.874258,29703355,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 15:47:08.186216,NaN,2026-03-03 17:57:39.22484,2026-04-30 23:22:30.324207,2026-06-26 07:07:08.332688,29703355,0.00
3,passenger_count,BIGINT,0,9,11,1.2494318488564,0.6529195072996764,1,1,1,29703355,25.98
4,trip_distance,DOUBLE,0.0,328522.2,7217,5.552940149689247,550.6497893231096,1.0239513653650012,1.8554594639170763,3.8139838813053486,29703355,0.00
5,RatecodeID,BIGINT,1,99,7,4.527471444398553,18.0009265400668,1,1,1,29703355,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,29703355,25.98
7,PULocationID,INTEGER,1,265,290,161.57793013617484,66.74656703251704,117,161,233,29703355,0.00
8,DOLocationID,INTEGER,1,265,298,161.05139342003622,70.72548569211344,109,162,234,29703355,0.00
9,payment_type,BIGINT,0,5,6,0.8621735154160195,0.6463324345961728,0,1,1,29703355,0.00


### Perfil estadístico de los taxis verdes

El mismo perfil para los taxis verdes.

In [12]:
taxi_db.show(con, "03_exploration/11_summary_green.sql")

SUMMARIZE SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.328351833504393,1.2771882973738604,2,2,2,337114,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 13:25:51.096427,NaN,2026-03-06 11:17:02.297791,2026-05-01 11:03:08.010598,2026-06-27 14:30:00.000092,337114,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 13:46:38.833759,NaN,2026-03-06 11:35:38.263634,2026-05-03 09:57:49.524605,2026-06-29 01:01:57.749535,337114,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,337114,14.47
4,RatecodeID,BIGINT,1,99,7,1.2549672434183374,1.0015327424296088,1,1,1,337114,14.47
5,PULocationID,INTEGER,1,265,263,97.32761024460568,56.57830275002367,74,75,104,337114,0.00
6,DOLocationID,INTEGER,1,265,266,142.8768458147689,77.24476821098119,75,140,229,337114,0.00
7,passenger_count,BIGINT,0,9,11,1.3006461144694266,0.9481007387491747,1,1,1,337114,14.47
8,trip_distance,DOUBLE,0.0,179830.92,2472,13.349507644298376,880.4035093234567,1.253016851675203,2.0652074810673624,3.6962840767568097,337114,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,4808,17.01410199517066,17.95862763527065,8.602896855429488,13.2028699522728,19.6581623355478,337114,0.00


### Reglas de calidad

Cuenta los viajes que violan cada regla de negocio, sobre la vista `trips` que une ambos tipos con nombres de columna comunes.

In [13]:
taxi_db.show(con, "03_exploration/12_quality_checks.sql")

WITH checks AS (
    SELECT
        taxi_type,
        count(*) AS total_trips,
        count(*) FILTER (
            WHERE pickup_datetime < make_timestamp(source_year, source_month, 1, 0, 0, 0)
               OR pickup_datetime >= make_timestamp(source_year, source_month, 1, 0, 0, 0) + INTERVAL 1 MONTH
        ) AS pickup_outside_file_month,
        count(*) FILTER (WHERE dropoff_datetime <= pickup_datetime) AS non_positive_duration,
        count(*) FILTER (WHERE dropoff_datetime > pickup_datetime + INTERVAL 6 HOUR) AS duration_over_6h,
        count(*) FILTER (WHERE trip_distance = 0) AS zero_distance,
        count(*) FILTER (WHERE trip_distance > 100) AS distance_over_100mi,
        count(*) FILTER (WHERE fare_amount < 0 OR total_amount < 0) AS negative_amount,
        count(*) FILTER (WHERE total_amount > 1000) AS total_over_1000,
        count(*) FILTER (WHERE passenger_count IS NULL) AS null_passenger_count,
        count(*) FILTER (WHERE passenger_count = 0) AS zero_passenger

,taxi_type,issue,affected_trips,pct_of_trips
0,yellow,null_passenger_count,7716688,25.979
1,yellow,payment_type_0,7716688,25.979
2,yellow,zero_distance,952231,3.206
3,yellow,ratecode_99,769693,2.591
4,yellow,non_positive_duration,371683,1.251
5,yellow,unknown_zone,201486,0.678
6,yellow,negative_amount,161970,0.545
7,yellow,zero_passengers,91359,0.308
8,yellow,duration_over_6h,7315,0.025
9,yellow,distance_over_100mi,1223,0.004


### Rango de fechas de abordaje

Comprueba si hay fechas fuera del periodo que cubre cada archivo.

In [14]:
taxi_db.show(con, "03_exploration/13_pickup_date_range.sql")

SELECT
    taxi_type,
    min(pickup_datetime) AS min_pickup,
    max(pickup_datetime) AS max_pickup,
    count(*) FILTER (WHERE year(pickup_datetime) < 2026) AS pickups_before_2026,
    count(*) FILTER (WHERE pickup_datetime > TIMESTAMP '2026-09-01') AS pickups_after_aug_2026
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type DESC;


,taxi_type,min_pickup,max_pickup,pickups_before_2026,pickups_after_aug_2026
0,yellow,2001-01-01 09:23:58,2026-08-31 23:59:59,17,0
1,green,2008-12-31 17:35:31,2026-08-31 23:58:28,14,0


### Patrón de valores faltantes

Relaciona los nulos de `passenger_count` con el tipo de pago.

In [15]:
taxi_db.show(con, "03_exploration/14_missing_values_pattern.sql")

SELECT
    taxi_type,
    payment_type,
    passenger_count IS NULL AS passenger_count_missing,
    count(*) AS trips,
    round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY taxi_type), 2) AS pct_of_type
FROM trips
GROUP BY taxi_type, payment_type, passenger_count IS NULL
ORDER BY taxi_type DESC, trips DESC;


,taxi_type,payment_type,passenger_count_missing,trips,pct_of_type
0,yellow,1,False,18941008,63.77
1,yellow,0,True,7716688,25.98
2,yellow,2,False,2708031,9.12
3,yellow,4,False,239488,0.81
4,yellow,3,False,98138,0.33
5,yellow,5,False,2,0.00
6,green,1,False,219980,65.25
7,green,2,False,65921,19.55
8,green,<NA>,True,48775,14.47
9,green,3,False,1688,0.50


### Registros duplicados

Agrupa por un hash de la fila completa para encontrar filas idénticas sin ordenar 30 millones de registros.

In [16]:
taxi_db.show(con, "03_exploration/15_duplicates.sql")

SELECT
    taxi_type,
    count(*) AS duplicated_groups,
    sum(copies - 1) AS extra_rows
FROM (
    SELECT taxi_type, row_hash, count(*) AS copies
    FROM (SELECT taxi_type, hash(t) AS row_hash FROM trips AS t)
    GROUP BY taxi_type, row_hash
    HAVING count(*) > 1
)
GROUP BY taxi_type
ORDER BY taxi_type DESC;


,taxi_type,duplicated_groups,extra_rows
0,yellow,7,7.0


### Duración por proveedor

Revisa si los viajes con duración cero o negativa se concentran en algún proveedor (`VendorID`).

In [17]:
taxi_db.show(con, "03_exploration/18_vendor_timestamps.sql")

SELECT
    taxi_type,
    VendorID AS vendor_id,
    count(*) AS trips,
    count(*) FILTER (WHERE dropoff_datetime = pickup_datetime) AS same_timestamp,
    count(*) FILTER (WHERE dropoff_datetime < pickup_datetime) AS dropoff_before_pickup,
    round(100.0 * count(*) FILTER (WHERE dropoff_datetime <= pickup_datetime) / count(*), 2) AS pct_non_positive_duration
FROM trips
GROUP BY ALL
ORDER BY taxi_type DESC, vendor_id;


,taxi_type,vendor_id,trips,same_timestamp,dropoff_before_pickup,pct_non_positive_duration
0,yellow,1,5467071,4297,4,0.08
1,yellow,2,23809774,256,2,0.00
2,yellow,6,59390,0,4,0.01
3,yellow,7,367120,367120,0,100.00
4,green,1,28696,86,0,0.30
5,green,2,273571,143,0,0.05
6,green,6,34847,0,5,0.01


### Columna nueva `request_source`

Distribución de la columna que aparece a partir de junio de 2026.

In [18]:
taxi_db.show(con, "03_exploration/16_request_source.sql")

SELECT
    taxi_type,
    source_month,
    coalesce(request_source, '(nulo)') AS request_source,
    count(*) AS trips
FROM trips
WHERE source_year = 2026
GROUP BY ALL
ORDER BY taxi_type DESC, source_month, trips DESC;


,taxi_type,source_month,request_source,trips
0,yellow,1,(nulo),3724889
1,yellow,2,(nulo),3399866
2,yellow,3,(nulo),3952451
3,yellow,4,(nulo),3831240
4,yellow,5,(nulo),4090836
5,yellow,6,(nulo),2824068
6,yellow,6,HV0003,927698
7,yellow,6,A,76711
8,yellow,6,EH0004,8107
9,yellow,6,CC,664


## Decisión: vista `trips_clean`

Con base en las reglas anteriores se definió la vista `trips_clean` (`sql/views/trips_clean.sql`), que descarta viajes fuera del mes del archivo, con duración no positiva (salvo el proveedor 7, que registra la misma hora de abordaje y descenso en todos sus viajes) o mayor a 6 horas, distancia 0 o mayor a 100 millas, tarifa negativa o total fuera de (0, 1000]. Los nulos de `passenger_count` **no** se descartan porque afectan a una cuarta parte de los viajes amarillos y el resto de sus campos es válido.

### Registros descartados por la limpieza

Compara el conteo de `trips` contra `trips_clean`.

In [19]:
taxi_db.show(con, "03_exploration/17_clean_vs_raw.sql")

SELECT
    r.taxi_type,
    r.trips AS raw_trips,
    c.trips AS clean_trips,
    r.trips - c.trips AS discarded_trips,
    round(100.0 * (r.trips - c.trips) / r.trips, 2) AS pct_discarded
FROM (SELECT taxi_type, count(*) AS trips FROM trips GROUP BY ALL) AS r
JOIN (SELECT taxi_type, count(*) AS trips FROM trips_clean GROUP BY ALL) AS c USING (taxi_type)
ORDER BY r.taxi_type DESC;


,taxi_type,raw_trips,clean_trips,discarded_trips,pct_discarded
0,yellow,29703355,28593831,1109524,3.74
1,green,337114,322606,14508,4.30


In [20]:
print(taxi_db.read_sql("views/trips.sql"))
print(taxi_db.read_sql("views/trips_clean.sql"))

CREATE OR REPLACE VIEW trips AS
SELECT
    'yellow' AS taxi_type,
    CAST(regexp_extract(filename, '_(\d{4})-\d{2}\.parquet$', 1) AS INTEGER) AS source_year,
    CAST(regexp_extract(filename, '_\d{4}-(\d{2})\.parquet$', 1) AS INTEGER) AS source_month,
    * EXCLUDE (filename) RENAME (
        tpep_pickup_datetime AS pickup_datetime,
        tpep_dropoff_datetime AS dropoff_datetime
    )
FROM yellow_raw
UNION ALL BY NAME
SELECT
    'green' AS taxi_type,
    CAST(regexp_extract(filename, '_(\d{4})-\d{2}\.parquet$', 1) AS INTEGER) AS source_year,
    CAST(regexp_extract(filename, '_\d{4}-(\d{2})\.parquet$', 1) AS INTEGER) AS source_month,
    * EXCLUDE (filename) RENAME (
        lpep_pickup_datetime AS pickup_datetime,
        lpep_dropoff_datetime AS dropoff_datetime
    )
FROM green_raw;

CREATE OR REPLACE VIEW trips_clean AS
SELECT
    *,
    CASE
        WHEN dropoff_datetime > pickup_datetime
            THEN date_diff('second', pickup_datetime, dropoff_datetime) / 60.0
    END AS